# 4. 多层感知机 (MLP) (下)

## 4.6 暂退法 (Dropout)

> 如果说“权重衰减”是通过限制参数的取值范围来防止过拟合，那么“暂退法”就是通过破坏网络结构的稳定性来强迫模型变得更健壮。

### 1. 核心动机：抑制“共谋” (Co-adaptation)

在深度学习中，过拟合不仅表现为参数过多，更表现为神经元之间的**共谋 (Co-adaptation)**。

#### 1.1 数学本质：脆弱的平衡
*   **现象**：在没有约束的情况下，模型为了最小化训练误差，可能会演化出一种“脆弱”的权重组合。例如，神经元 $h_1$ 捕捉了一个噪声特征，而神经元 $h_2$ 通过特定的权重分配恰好抵消了 $h_1$ 引入的误差。
*   **数学解释**：这意味着权重矩阵的各个维度之间存在强烈的**协方差 (Covariance)**。这种复杂的相互依赖仅在训练集上有效，一旦遇到未见的测试数据，这种平衡就会崩溃，导致泛化能力极差。
*   **解决思路**：如果我们在训练时随机“抹除”一部分神经元，迫使每个神经元在失去“搭档”的情况下依然能提取有用的特征。

---

### 2. 数学定义与机制

#### 2.1 随机扰动公式
假设隐藏层的输出向量为 $\mathbf{h}$。Dropout 以概率 $p$ 将其元素置为 0，以 $1-p$ 的概率保留并缩放。
对于每个元素 $h_i$，经过 Dropout 后的 $h_i'$ 为：

$$
h_i' = \begin{cases} 0 & \text{概率为 } p \\ \frac{h_i}{1-p} & \text{概率为 } 1-p \end{cases}
$$

#### 2.2 期望值不变性 (Invariance of Expectation)
Dropout 设计的核心在于**无偏性**。为了确保训练和测试时下一层接收到的信号能量一致，必须执行缩放：
$$E[h_i'] = p \cdot 0 + (1-p) \cdot \frac{h_i}{1-p} = h_i$$
**结论**：Dropout 注入的是**乘性噪声**，但在统计意义上保持了输出的期望值不变。

#### 2.3 隐式正则化证明 (Linear Case)
对于线性模型，可以证明最小化 Dropout 后的期望损失等价于：
$$J(\mathbf{w}) = \text{Loss}_{original} + \underbrace{\frac{p}{2(1-p)} \sum_{i} w_i^2 x_i^2}_{\text{隐式 } L_2 \text{ 正则项}}$$
**直觉**：Dropout 强制让模型惩罚那些过大的权重 $w_i$，因为权重越大，随机丢弃带来的波动（方差）就越剧烈。为了收敛，模型必须选择更平滑、更简单的参数。

---

### 3. 关键状态管理：Train vs. Eval

这是 Dropout 实验中最容易产生 Bug 的地方：

| 模式 | 标志位 | Dropout 行为 | 目的 |
| :--- | :--- | :--- | :--- |
| **训练模式** | `net.train()` | **开启**随机丢弃 | 引入噪声，防止神经元共谋，增强泛化。 |
| **评估模式** | `net.eval()` | **关闭**（Identity） | 使用所有神经元的组合力量进行确定性预测。 |

**工程规范建议**：在执行任何评估逻辑（如计算测试集准确率）之前，**必须**显式调用 `net.eval()`，否则测试结果会因为随机性产生剧烈抖动。

---

### 4. 经验法则与暗知识

1.  **放置位置**：通常放在**激活函数 (ReLU) 之后**。
2.  **丢弃率选择**：
    *   靠近输入层的层：$p$ 较小（如 0.2），保护原始特征。
    *   靠近输出层的深层：$p$ 较大（如 0.5），抑制复杂的抽象共谋。
3.  **学习率调整**：使用 Dropout 后，模型收敛变慢，通常可以适当**调大**学习率（$lr$）或增加训练轮数（$epochs$）。
4.  **与权重衰减配合**：Dropout 限制了神经元间的“配合”，L2 限制了权重的“大小”。两者结合是目前对抗过拟合的标准配置。

---

### 5. 总结
**暂退法 (Dropout)** 通过在训练中制造“信息缺失”，迫使神经网络学习到更具普适性的特征组合。它在数学上等价于一种自适应的正则化，在工程上是提高深度模型鲁棒性的核心工具。


In [10]:
# 暂退法的从零实现。
# 注意：Dropout 只在“训练模式”下生效。
import torch
from torch import Tensor, nn
from typing import Optional

def dropout_layer(X: Tensor, dropout_prob: float) -> Tensor:
    """手动实现 Dropout 层。
    
    Args:
        X: 输入张量。
        dropout_prob: 丢失概率 p (0<= p <= 1)。

    Returns:
        经过随机遮罩并缩放后的张量。
    """
    # 1. 边界情况处理
    if dropout_prob == 1.0:
        return torch.zeros_like(X)
    if dropout_prob == 0.0:
        return X
    
    # 2. 生成遮罩 (Mask)
    # torch.rand 生成 [0, 1) 区间的随机浮点数
    # 判断 > p 得到布尔矩阵
    # 转换成 float 后，保留的位置是 1，丢弃的是 0
    mask: Tensor = (torch.rand_like(X) > dropout_prob).float()

    # 3. 元素相乘并执行期望修正 (Scaling)
    return (mask * X) / (1.0 - dropout_prob)

# 测试函数
X_test = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print(f"原始数据:\n{X_test}")
print(f"p=0.5 时的 Dropout 输出:\n{dropout_layer(X_test, 0.5)}")

原始数据:
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
p=0.5 时的 Dropout 输出:
tensor([[ 0.,  0.,  4.,  0.,  0.,  0., 12.,  0.],
        [16., 18.,  0., 22.,  0., 26., 28., 30.]])


> 这个 Python 注释竟然支持 markdown 语法。

In [11]:
# 带有 Dropout 的多层感知机
# 该实验用的数据集是那个 Fashion-MNIST。
class DropoutMLPScratch(nn.Module):
    """从零实现带 Dropout 的多层感知机 (两个隐藏层)，即简洁实现的 nn.Sequential 容器。
    
    **继承 nn.Module 的关键特性**:
        1. **参数管理** - `nn.Module` 自动追踪所有子模块中的参数 (权重和偏置)。
        2. **训练/评估模式** - 通过 `.trian()` 和 `.eval()` 方法切换，Dropout 会根据模式自动启用或禁用。
        3. **梯度计算** - 自动支持反向传播和梯度更新。
        4. **模型移动** - 支持 `.to(device)` 将整个模型移到 GPU/CPU。
        5. **参数访问** - 可以通过 `.parameters()`、`.named_parameters()` 等方法访问所有参数。
        6. **子模块注册** - 在 `__init__` 中定义的子模块会自动注册。
    """

    def __init__(self, num_inputs: int, num_outputs: int, num_hiddens1: int,
                 num_hiddens2: int, dropout1: float, dropout2: float):
        super().__init__()
        self.num_inputs = num_inputs

        # 定义层参数
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()

        self.dropout1 = dropout1
        self.dropout2 = dropout2

    def forward(self, X: Tensor) -> Tensor:
        """前向传播 (即定义的模型)。"""

        # 1. 展平输入
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))
        
        # 2. 第一层 Dropout: 仅在训练模式下起效。
        if self.training:
            H1 = dropout_layer(H1, self.dropout1)

        H2 = self.relu(self.lin2(H1))
        
        # 3. 第二层 Dropout
        if self.training:
            H2 = dropout_layer(H2, self.dropout2)

        return self.lin3(H2)

In [12]:
# 暂退法的简洁实现
def get_dropout_net(
    num_inputs: int,
    num_outputs: int,
    num_hiddens1: int,
    num_hiddens2: int,  
    dropout1: float,
    dropout2: float 
) -> nn.Sequential:
    """使用 PyTorch 内置层构建带 Dropout 的 MLP。"""

    net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(num_inputs, num_hiddens1),
        nn.ReLU(),
        # 插入 Dropout 层
        nn.Dropout(dropout1),
        nn.Linear(num_hiddens1, num_hiddens2),
        nn.ReLU(),
        nn.Dropout(dropout2),
        nn.Linear(num_hiddens2, num_outputs)
    )

    # 统一初始化
    def init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.01)
            if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    net.apply(init_weights)
    return net

**丢弃率 $p$ 通常设为多少？**
*   通常在 `0.2` 到 `0.5` 之间。
*   **经验**：靠近输入层的层可以设小一点（比如 0.2），靠近输出层的深层可以设大一点（比如 0.5）。

In [13]:
# 实验参数配置
from dataclasses import dataclass

@dataclass(frozen=True)
class MLPConfig:
    num_inputs: int = 784
    num_outputs: int = 10
    num_hiddens1: int = 256
    num_hiddens2: int = 256
    dropout1: float = 0.0       # 默认不开启
    dropout2: float = 0.0
    lr: float = 0.5             # Dropout 训练通常可以用大一点的学习率
    num_epochs: int = 10
    batch_size: int = 256

In [14]:
# 开始实验
import d2l_utils as d2l

# 1. 加载数据集
train_iter, test_iter = d2l.load_fashion_mnist(batch_size=256)
loss_fn = nn.CrossEntropyLoss(reduction='none')

# 2. 定义实验 A: 无 Dropout (过拟合对照组)
config_none = MLPConfig()
net_none = get_dropout_net(
    config_none.num_inputs,
    config_none.num_outputs,
    config_none.num_hiddens1,
    config_none.num_hiddens2,
    config_none.dropout1,
    config_none.dropout2
)
train_none = torch.optim.SGD(net_none.parameters(), lr=config_none.lr)

print(">>> 开始训练：无 Dropout 模型")
d2l.train_softmax(net_none, train_iter, test_iter, loss_fn, config_none.num_epochs, train_none)

# 3. 定义实验 B: 有 Dropout (手写实现组、history_dropout2 =  简化实现组)
config_dropout = MLPConfig(dropout1=0.2, dropout2=0.5)
net_dropout1 = DropoutMLPScratch(
    config_dropout.num_inputs,
    config_dropout.num_outputs,
    config_dropout.num_hiddens1,
    config_dropout.num_hiddens2,
    config_dropout.dropout1,
    config_dropout.dropout2
)
net_dropout2 = get_dropout_net(
    config_dropout.num_inputs,
    config_dropout.num_outputs,
    config_dropout.num_hiddens1,
    config_dropout.num_hiddens2,
    config_dropout.dropout1,
    config_dropout.dropout2
)
train_dropout1 = torch.optim.SGD(net_dropout1.parameters(), config_dropout.lr)
train_dropout2 = torch.optim.SGD(net_dropout2.parameters(), config_dropout.lr)
print("\n>>> 开始训练：带 Dropout 模型 (Scratch)")
d2l.train_softmax(net_dropout1, train_iter, test_iter, loss_fn, config_dropout.num_epochs, train_dropout1)

print("\n>>> 开始训练：带 Dropout 模型 (Concise)")
d2l.train_softmax(net_dropout2, train_iter, test_iter, loss_fn, config_dropout.num_epochs, train_dropout2)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
>>> 开始训练：无 Dropout 模型
开始训练，总轮数: 10。
Epoch 1: Loss = 1.2559, Train Acc = 0.5197, Test Acc = 0.6514
Epoch 2: Loss = 0.5866, Train Acc = 0.7732, Test Acc = 0.8087
Epoch 3: Loss = 0.4710, Train Acc = 0.8227, Test Acc = 0.8145
Epoch 4: Loss = 0.4198, Train Acc = 0.8433, Test Acc = 0.8411
Epoch 5: Loss = 0.3846, Train Acc = 0.8555, Test Acc = 0.8439
Epoch 6: Loss = 0.3628, Train Acc = 0.8633, Test Acc = 0.8179
Epoch 7: Loss = 0.3483, Train Acc = 0.8688, Test Acc = 0.8096
Epoch 8: Loss = 0.3346, Train Acc = 0.8744, Test Acc = 0.8046
Epoch 9: Loss = 0.3208, Train Acc = 0.8786, Test Acc = 0.8578
Epoch 10: Loss = 0.3049, Train Acc = 0.8852, Test Acc = 0.8331
训练完成！

>>> 开始训练：带 Dropout 模型 (Scratch)
开始训练，总轮数: 10。
Epoch 1: Loss = 0.8849, Train Acc = 0.6693, Test Acc = 0.7197
Epoch 2: Loss = 0.5303, Train Acc = 0.8041, Test Acc = 0.7503
Epoch 3: Loss = 0.4683, Train Acc = 0.8278, Test Acc = 0.8309
Epoch 4: Loss = 0.4222, Train A

## 4.7 前向传播、反向传播和计算图